# Django, Views

## Introduction to Django Views
---

In this lesson, you'll learn how to write **Django views** - the heart of your application's logic.

Views are Python functions or classes that receive HTTP requests and return HTTP responses.

1. [Function-Based Views (FBV)](#Function-Based-Views-(FBV)),
    - [Basic Structure](#Basic-Structure),
    - [Returning Different Response Types](#Returning-Different-Response-Types),
2. [Request and Response Objects](#Request-and-Response-Objects),
    - [The Request Object](#The-Request-Object),
    - [The Response Object](#The-Response-Object),
3. [Class-Based Views (CBV)](#Class-Based-Views-(CBV)),
    - [Basic CBV Structure](#Basic-CBV-Structure),
    - [HTTP Method Handlers](#HTTP-Method-Handlers),
4. [FBV vs CBV - When to Use Which](#FBV-vs-CBV---When-to-Use-Which),
    - [Comparison](#Comparison),
    - [Practical Guidelines](#Practical-Guidelines),
5. [🧠 EXERCISE 🧠, Create Views for Bookstore](#🧠-EXERCISE-🧠,-Create-Views-for-Bookstore).

<br>

## Function-Based Views (FBV)

---

### Basic Structure

---

A **function-based view** (FBV) is simply a Python function that:

1. Takes an `HttpRequest` object as the first argument
2. Returns an `HttpResponse` object

<br>

```
Request  →  View Function  →  Response
```

In [ ]:
# Example: Simplest possible view

from django.http import HttpRequest, JsonResponse
from blog.models import Blog

def blog_list(request: HttpRequest):
    blogs = Blog.objects.all()
    return JsonResponse(list(blogs.values()), safe=False)

In [ ]:
# Example: View with URL parameter

from django.shortcuts import get_object_or_404
from django.http import JsonResponse

def blog_detail(request: HttpRequest, pk: int):
    blog = get_object_or_404(Blog, pk=pk)
    return JsonResponse({
        'id': blog.id,
        'title': blog.title,
        'author': blog.author,
        'published_date': blog.published_date.isoformat()
    })

# URL pattern: path('blogs/<int:pk>/', views.blog_detail)
# Visit: /blogs/1/  →  "{"id": 1, "title": "My First Blog", "author": "John Doe", "published_date": "2021-01-01"}"

But you can use even more parameters.

<br>

### Returning Different Response Types

---

Views can return different types of responses:

In [ ]:
# Example: Different response types

from django.http import HttpResponse, JsonResponse, HttpResponseRedirect, Http404
from django.shortcuts import render, redirect, get_object_or_404

# 1. Plain text
def plain_text(request):
    return HttpResponse("Just plain text", content_type="text/plain")

# 2. HTML
def html_response(request):
    return HttpResponse("<h1>HTML Heading</h1>")

# 3. JSON (for APIs)
def json_response(request):
    data = {'name': 'Django', 'version': '5.0'}
    return JsonResponse(data)

# 4. Redirect
def redirect_view(request):
    return redirect('book_list')  # Redirects to named URL

In [ ]:
# Example: Rendering templates (most common)

from django.shortcuts import render

def blog_list_example(request):
    """Display list of blogs using a template."""
    blogs = [
        {'id': 1, 'title': 'Django for Beginners', 'author': 'William Vincent'},
        {'id': 2, 'title': 'Two Scoops of Django', 'author': 'Daniel Feldroy'},
        {'id': 3, 'title': 'Django for Professionals', 'author': 'William Vincent'},
    ]
    return render(request, 'blog/blog_list_dynamic.html', {'blogs': blogs})

In [ ]:
# Example: Handling errors

from django.http import Http404, HttpResponseForbidden
from django.shortcuts import get_object_or_404

def blog_detail(request: HttpRequest, pk: int):
    blog = get_object_or_404(Blog, pk=pk)
    
    # TODO
    if pk not in blog:
        # Option 1: Raise Http404 exception
        raise Http404(f"Blog {pk} not found")
    
    return HttpResponse(f"<h1>{blog[pk]}</h1>")

def protected_view(request):
    """Return 403 Forbidden for unauthorized access."""
    if not request.user.is_staff:
        return HttpResponseForbidden("Staff only!")
    return HttpResponse("Welcome, staff member!")

You use this condition most often when you want to hide something from regular visitors but show it to site admins.

<br>

## Request and Response Objects

---

### The Request Object

---

The `HttpRequest` object contains all information about the incoming request:

<br>

| Attribute | Description | Example |
|-----------|-------------|---------|
| `request.method` | HTTP method | `'GET'`, `'POST'` |
| `request.path` | URL path | `'/books/5/'` |
| `request.GET` | Query parameters | `{'page': '2'}` |
| `request.POST` | POST data | `{'title': 'New Book'}` |
| `request.user` | Current user | `<User: alice>` |
| `request.session` | Session data | `{'cart': [...]}` |
| `request.COOKIES` | Cookies | `{'csrftoken': '...'}` |
| `request.META` | HTTP headers | `{'HTTP_HOST': '...'}` |

In [ ]:
# Example: Using request attributes

from django.http import HttpRequest, HttpResponse

def debug_request(request: HttpRequest) -> HttpResponse:
    """Show request information for debugging."""
    info = f"""
    Method: {request.method}
    Path: {request.path}
    Full URL: {request.build_absolute_uri()}
    User: {request.user}
    Is authenticated: {request.user.is_authenticated}
    """
    return HttpResponse(f"<pre>{info}</pre>")

# Example: Handling POST data - create a new blog

from django.http import HttpRequest, HttpResponse, JsonResponse
from django.views.decorators.http import require_http_methods
from django.views.decorators.csrf import csrf_exempt
from blog.models import Blog
import json

@csrf_exempt  # Only for demo/curl testing - in production use CSRF tokens
@require_http_methods(["GET", "POST"])
def blog_create(request: HttpRequest) -> HttpResponse:
    """Create a new blog via POST, or show form via GET."""

    if request.method == 'POST':
        # Support both form data and JSON body
        if request.content_type == 'application/json':
            data = json.loads(request.body)
        else:
            data = request.POST

        title = data.get('title', '')
        author = data.get('author', '')
        published_date = data.get('published_date', '')

        if title and author and published_date:
            blog = Blog.objects.create(
                title=title,
                author=author,
                published_date=published_date,
            )
            return JsonResponse({'id': blog.id, 'title': blog.title}, status=201)
        else:
            return JsonResponse({'error': 'title, author and published_date are required'}, status=400)

    # GET - show a simple HTML form
    return HttpResponse("""
        <form method="post">
            <input name="title" placeholder="Title"><br>
            <input name="author" placeholder="Author"><br>
            <input name="published_date" placeholder="YYYY-MM-DD"><br>
            <button type="submit">Create</button>
        </form>
    """)

# URL pattern: path('new/', views.blog_create, name='blog_create')
# Test with curl:
# curl -X POST http://localhost:8000/blog/new/ \
#      -H "Content-Type: application/json" \
#      -d '{"title": "My Blog", "author": "John", "published_date": "2024-01-01"}'

In [ ]:
# Example: Working with query parameters

from django.http import HttpRequest, HttpResponse, JsonResponse
from blog.models import Blog

def search_blogs(request: HttpRequest) -> JsonResponse:
    """Search blogs with query parameters."""
    # URL: /blogs/search/?q=django&sort=title
    
    query = request.GET.get('q', '')          # 'django' or empty string
    sort = request.GET.get('sort', 'title')   # 'title' (default)
    
    blogs = Blog.objects.all()
    
    if query:
        blogs = blogs.filter(title__icontains=query)
    
    if sort in ('title', 'author', 'published_date'):
        blogs = blogs.order_by(sort)
    
    return JsonResponse(list(blogs.values()), safe=False)

# URL pattern: path('search/', views.search_blogs, name='blog_search')
# Visit: /blogs/search/?q=django&sort=published_date

- **title:** The name of the database field you are searching in (e.g., the blog title).

- **i:** Stands for "insensitive." It ignores the difference between uppercase A and lowercase a (case-insensitive).

- **contains:** Means "contains." It looks for that specific snippet of text anywhere within a word or a sentence.

In [ ]:
# Example: Handling POST data

import json

from http import HTTPStatus
from django.http import HttpRequest, HttpResponse
from django.views.decorators.http import require_http_methods

@csrf_exempt  # Only for demo/curl testing - in production use CSRF tokens
@require_http_methods(["GET", "POST"])
def blog_create(request: HttpRequest) -> HttpResponse:
    """Create a new blog via POST, or show form via GET."""

    if request.method == 'POST':
        # Support both form data and JSON body
        if request.content_type == 'application/json':
            data = json.loads(request.body)
        else:
            data = request.POST

        title = data.get('title', '')
        author = data.get('author', '')
        published_date = data.get('published_date', '')

        if title and author and published_date:
            blog = Blog.objects.create(
                title=title,
                author=author,
                published_date=published_date,
            )
            return JsonResponse({'id': blog.id, 'title': blog.title}, status=HTTPStatus.CREATED)
        else:
            return JsonResponse(
                {'error': 'title, author and published_date are required'},
                status=HTTPStatus.BAD_REQUEST
            )

    # GET - show a simple HTML form
    return JsonResponse(
        {
            "id": None,
            "title": None,
            "author": None,
            "published_date": None,
        },
        status=HTTPStatus.OK
    )

# Test with curl:
# curl -X POST http://localhost:8000/blog/new/ \
#      -H "Content-Type: application/json" \
#      -d '{"title": "My Blog", "author": "John", "published_date": "2024-01-01"}'

`@csrf_exempt` This decorator tells Django to skip the security check for this specific view. It’s like leaving the back door unlocked so you can test your code easily with tools like curl without needing a security token.


`@require_http_methods` This acts as a filter that only allows specific types of requests, like GET or POST. If someone tries to access the view using a different method, Django will automatically block them.


In [ ]:
from django.urls import path
from . import views

app_name = 'blog'

urlpatterns = [
    path('', views.blog_list, name='blog_list'),
    path('new/', views.blog_create, name='blog_create'),
    path('<int:pk>', views.blog_detail, name='blog_detail'),
    path('search/', views.search_blogs, name='blog_search'),
    path('examples/', views.book_list_example, name='book_list_example'),
    path('author/<str:author_name>/', views.blogs_by_author, name='blogs_by_author'),
]

<br>

### The Response Object

---

The `HttpResponse` object represents the response you send back:

<br>

| Method/Attribute | Description |
|------------------|-------------|
| `HttpResponse(content)` | Create response with content |
| `response.status_code` | HTTP status code (200, 404, etc.) |
| `response['Header']` | Set response headers |
| `response.set_cookie()` | Set a cookie |

In [ ]:
# Example: Customizing responses

from django.http import HttpResponse

def custom_response(request):
    """Return a customized response."""
    response = HttpResponse("Custom response")
    
    # Set status code
    response.status_code = 200
    
    # Set headers
    response['X-Custom-Header'] = 'Hello'
    response['Content-Type'] = 'text/plain'
    
    # Set cookie
    response.set_cookie('visited', 'true', max_age=3600)  # 1 hour
    
    return response

In [ ]:
# Example: Common response shortcuts

from django.http import (
    HttpResponse,           # 200 OK
    HttpResponseRedirect,   # 302 Found (redirect)
    HttpResponseNotFound,   # 404 Not Found
    HttpResponseForbidden,  # 403 Forbidden
    HttpResponseBadRequest, # 400 Bad Request
    JsonResponse,           # JSON with correct content-type
)

# Or use status parameter
def created_response(request):
    return HttpResponse("Resource created", status=201)

def no_content_response(request):
    return HttpResponse(status=204)

[List of status codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status#informational_responses)
```
# Full overview of common HTTP status codes:
#
# 2xx - Success
#   200 OK              - standard successful response
#   201 Created         - resource was created (POST)
#   204 No Content      - success, no body (DELETE)
#
# 3xx - Redirection
#   301 Moved Permanently - URL changed forever
#   302 Found             - temporary redirect
#
# 4xx - Client errors
#   400 Bad Request     - invalid input from client
#   401 Unauthorized    - not authenticated
#   403 Forbidden       - authenticated but not allowed
#   404 Not Found       - resource doesn't exist
#   405 Method Not Allowed - wrong HTTP method
#
# 5xx - Server errors
#   500 Internal Server Error - unhandled exception in view
```

<br>

## Class-Based Views (CBV)

---

### Basic CBV Structure

---

A **class-based view** (CBV) is a Python class that handles HTTP requests:

- Inherits from `View` (or its subclasses)
- Defines methods for HTTP verbs: `get()`, `post()`, `put()`, `delete()`, etc.
- Uses `.as_view()` in URL configuration

In [ ]:
# Example: Basic class-based view

from django.http import HttpRequest, HttpResponse
from django.views import View

class HelloView(View):
    """A simple class-based view."""
    
    def get(self, request: HttpRequest) -> HttpResponse:
        """Handle GET requests."""
        return HttpResponse("Hello from CBV!")

# In urls.py:
# path('hello/', HelloView.as_view(), name='hello')

In [ ]:
# Example: CBV with URL parameter

from django.http import HttpRequest, HttpResponse, JsonResponse
from django.views import View
from django.shortcuts import get_object_or_404
from blog.models import Blog

class BlogDetailView(View):
    """Display blog details."""
    
    def get(self, request: HttpRequest, pk: int) -> HttpResponse:
        """Handle GET request for blog detail."""
        # URL parameters are passed as keyword arguments
        blog = get_object_or_404(Blog, pk=pk)
        return JsonResponse({
            'id': blog.id,
            'title': blog.title,
            'author': blog.author,
            'published_date': blog.published_date.isoformat(),
        })

# In urls.py:
# path('blogs/<int:pk>/', BlogDetailView.as_view(), name='blog_detail')

<br>

### HTTP Method Handlers

---

CBVs handle different HTTP methods with separate methods:

| HTTP Method | CBV Method |
|-------------|------------|
| GET | `get(self, request, *args, **kwargs)` |
| POST | `post(self, request, *args, **kwargs)` |
| PUT | `put(self, request, *args, **kwargs)` |
| PATCH | `patch(self, request, *args, **kwargs)` |
| DELETE | `delete(self, request, *args, **kwargs)` |

In [ ]:
from django.utils.decorators import method_decorator
from django.views.decorators.csrf import csrf_exempt

@method_decorator(csrf_exempt, name='dispatch')
class BlogCreateView(View):
    """Create a new blog via POST, or show form via GET."""

    def get(self, request: HttpRequest) -> JsonResponse:
        return JsonResponse(
            {
                "id": None,
                "title": None,
                "author": None,
                "published_date": None,
            },
            status=HTTPStatus.OK
        )

    def post(self, request: HttpRequest) -> JsonResponse:
        if request.content_type == 'application/json':
            data = json.loads(request.body)
        else:
            data = request.POST

        title = data.get('title', '')
        author = data.get('author', '')
        published_date = data.get('published_date', '')

        if title and author and published_date:
            blog = Blog.objects.create(
                title=title,
                author=author,
                published_date=published_date,
            )
            return JsonResponse({'id': blog.id, 'title': blog.title}, status=HTTPStatus.CREATED)
        else:
            return JsonResponse(
                {'error': 'title, author and published_date are required'},
                status=HTTPStatus.BAD_REQUEST
            )

Class-Based Views (CBVs) work differently than regular functions. You can't just put `@csrf_exempt` on a class because it's designed for functions.

That’s why we use `@method_decorator`. It tells Django: 'Take this function decorator (csrf_exempt) and apply it to a specific method inside this class.'

We target the `dispatch` method because it’s the entry point for every request. By exempting dispatch, we essentially turn off the security check for the entire class.

<br>

With CBV you also need to update the `urls.py` module.

In [ ]:
from django.urls import path
from . import views

app_name = 'blog'

urlpatterns = [
    # ...
    path('new/', views.BlogCreateView.as_view(), name='blog_create'),
    path('<int:pk>', views.BlogDetailView.as_view(), name='blog_detail'),
    # ...
]

In [ ]:
# Example: CBV with shared setup logic

from django.http import HttpRequest, HttpResponse
from django.views import View

class BlogView(View):
    
    # Class attribute - shared data
    blogs = {
        1: {'title': 'Django Basics', 'author': 'Matous'},
        2: {'title': 'Advanced Django', 'author': 'David'},
    }
    
    def setup(self, request: HttpRequest, *args, **kwargs):
        """Called before dispatch - initialize shared state."""
        super().setup(request, *args, **kwargs)
        # Any setup logic here
    
    def get(self, request: HttpRequest, pk: int) -> HttpResponse:
        ...
    
    def delete(self, request: HttpRequest, pk: int) -> HttpResponse:
        ...

In [ ]:
# Example: Using TemplateView for simple template rendering

from django.views.generic import TemplateView
from blog.models import Blog

class BlogListTemplateView(TemplateView):
    """Display list of blogs using a template."""
    template_name = 'blog/blog_list.html'

    def get_context_data(self, **kwargs):
        """Add blogs to the template context."""
        context = super().get_context_data(**kwargs)
        context['blogs'] = Blog.objects.all()
        context['title'] = 'All Blog Posts'
        return context

# Even simpler - directly in urls.py:
# path('blogs/', BlogListTemplateView.as_view(), name='blog_list_template')

In [ ]:
  # TemplateView přímo v urls.py (bez views.py)
    path('about/', TemplateView.as_view(template_name='blog/about.html'), name='about'),

<br>

## FBV vs CBV - When to Use Which

---

### Comparison

---

| Aspect | Function-Based Views | Class-Based Views |
|--------|---------------------|-------------------|
| **Simplicity** | ✅ Easy to understand | ❌ More complex |
| **Explicit** | ✅ All logic visible | ❌ Hidden in inheritance |
| **Reusability** | ❌ Harder to reuse | ✅ Inheritance, mixins |
| **HTTP methods** | Manual if/else | Separate methods |
| **Decorators** | Simple to apply | Use `method_decorator` |
| **Testing** | Straightforward | May need more setup |

In [ ]:
# Comparison: Same functionality - FBV vs CBV

# === FBV approach ===
from django.http import HttpResponse
from django.views.decorators.http import require_http_methods

@require_http_methods(["GET", "POST"])
def blog_form_fbv(request, pk=None):
    if request.method == 'GET':
        # Show form
        return HttpResponse("Showing blog form")
    else:  # POST
        # Process form
        return HttpResponse("Processing blog form")

In [ ]:
# === CBV approach ===
from django.http import HttpResponse
from django.views import View

class BlogFormView(View):
    def get(self, request, pk=None):
        """Show the form."""
        return HttpResponse("Showing blog form")
    
    def post(self, request, pk=None):
        """Process the form."""
        return HttpResponse("Processing blog form")

<br>

### Practical Guidelines

---

**Use FBV when:**

- Simple, one-off views
- Learning Django (easier to understand)
- Views with complex, custom logic
- Single HTTP method handling

<br>

**Use CBV when:**

- CRUD operations (use generic views)
- Need to reuse code across views
- Multiple HTTP methods with shared setup
- Following RESTful patterns

In [ ]:
# Example: When FBV is cleaner

from django.http import JsonResponse

def health_check(request):
    """Simple health check endpoint."""
    return JsonResponse({'status': 'ok'})

# No need for a class here - it would be overkill

In [ ]:
# Example: When CBV shines - reusable base class

from django.views import View
from django.http import JsonResponse

class APIView(View):
    """Base class for API views."""
    
    def dispatch(self, request, *args, **kwargs):
        """Add common API logic."""
        # Check API key, rate limiting, etc.
        return super().dispatch(request, *args, **kwargs)
    
    def json_response(self, data, status=200):
        """Helper method for JSON responses."""
        return JsonResponse(data, status=status)


class BookAPIView(APIView):
    """Book API - inherits common API logic."""
    
    def get(self, request, pk=None):
        if pk:
            return self.json_response({'id': pk, 'title': 'Book'})
        return self.json_response({'books': []})


class AuthorAPIView(APIView):
    """Author API - also inherits common API logic."""
    
    def get(self, request, pk=None):
        return self.json_response({'authors': []})

<br>

## Useful View Decorators and Shortcuts

---

In [ ]:
# Common decorators for FBVs

from django.views.decorators.http import require_GET, require_POST, require_http_methods
from django.views.decorators.csrf import csrf_exempt
from django.contrib.auth.decorators import login_required

@require_GET
def list_view(request):
    """Only accepts GET requests."""
    return HttpResponse("List")

@require_POST
def create_view(request):
    """Only accepts POST requests."""
    return HttpResponse("Created")

@require_http_methods(["GET", "POST"])
def form_view(request):
    """Accepts GET and POST only."""
    return HttpResponse("Form")

@login_required
def protected_view(request):
    """Requires user to be logged in."""
    return HttpResponse(f"Hello, {request.user}!")

In [ ]:
# Common shortcuts

from django.shortcuts import render, redirect, get_object_or_404, get_list_or_404

def book_detail(request, pk):
    """Using get_object_or_404."""
    # Raises Http404 if not found
    # book = get_object_or_404(Book, pk=pk)
    # return render(request, 'book_detail.html', {'book': book})
    pass

def redirect_example(request):
    """Using redirect shortcut."""
    # Redirect to named URL
    return redirect('catalog:book_list')
    
    # Or to a URL path
    # return redirect('/books/')
    
    # Or to an object with get_absolute_url()
    # return redirect(book)

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **FBV** | Simple function: `def view(request)` → `HttpResponse` |
| **CBV** | Class with methods: `get()`, `post()`, etc. |
| **Request object** | `request.method`, `.GET`, `.POST`, `.user` |
| **Response object** | `HttpResponse`, `JsonResponse`, `redirect()` |
| **render()** | Shortcut to render template with context |
| **URL parameters** | Passed as function/method arguments |
| **When FBV** | Simple views, custom logic, learning |
| **When CBV** | CRUD, reusability, multiple HTTP methods |

<br>

### 🧠 EXERCISE 🧠, Create Views for Bookstore

---

Using the `mysite` project:

1. Create a FBV `blog_list` that returns a list of books as JSON
2. Create a FBV `blog_detail` that takes `pk` and returns book info or 404
3. Create a CBV `BlogSearchView` that:
   - GET: returns search form (just text for now)
   - POST: returns search results based on `request.POST.get('query')`
4. Add URL patterns for all views
5. Test all endpoints with the browser and/or curl

<details>
    <summary>▶️ Solution</summary>

**1. Add to `blog/views.py`:**

```python
def debug_request(request):
    """Display debug information about the request."""
    info = {
        'method': request.method,
        'path': request.path,
        'query_params': dict(request.GET),
        'user': str(request.user),
        'is_authenticated': request.user.is_authenticated,
        'user_agent': request.META.get('HTTP_USER_AGENT', 'Unknown'),
    }
    return JsonResponse(info, json_dumps_params={'indent': 2})
```

**2. Add to `blog/urls.py`:**

```python
urlpatterns = [
    # ... existing patterns ...
    path('debug/', views.debug_request, name='debug'),
]
```

**3. Test:**

```bash
curl "http://127.0.0.1:8000/blog/debug/?foo=bar&page=2"
```

Expected output:
```json
{
  "method": "GET",
  "path": "/blog/debug/",
  "query_params": {"foo": ["bar"], "page": ["2"]},
  "user": "AnonymousUser",
  "is_authenticated": false,
  "user_agent": "curl/7.68.0"
}
```
</details>

<br>

### 🧠 EXERCISE 🧠, Add Request Information View

---

Create a debug view that displays request information:

1. Create a FBV `debug_request` that returns:
   - HTTP method
   - Path
   - Query parameters (if any)
   - User (authenticated or anonymous)
   - User-Agent header
2. Add URL pattern at `/debug/`
3. Test with: `http://127.0.0.1:8000/debug/?foo=bar&page=2`

<details>
    <summary>▶️ Solution</summary>

**1. Add to `blog/views.py`:**

```python
def debug_request(request):
    """Display debug information about the request."""
    info = {
        'method': request.method,
        'path': request.path,
        'query_params': dict(request.GET),
        'user': str(request.user),
        'is_authenticated': request.user.is_authenticated,
        'user_agent': request.META.get('HTTP_USER_AGENT', 'Unknown'),
    }
    return JsonResponse(info, json_dumps_params={'indent': 2})
```

**2. Add to `blog/urls.py`:**

```python
urlpatterns = [
    # ... existing patterns ...
    path('debug/', views.debug_request, name='debug'),
]
```

**3. Test:**

```bash
curl "http://127.0.0.1:8000/blog/debug/?foo=bar&page=2"
```

Expected output:
```json
{
  "method": "GET",
  "path": "/blog/debug/",
  "query_params": {"foo": ["bar"], "page": ["2"]},
  "user": "AnonymousUser",
  "is_authenticated": false,
  "user_agent": "curl/7.68.0"
}
```
</details>

---